# 🎻 Kochel - Training Token Classification & ONNX Export

Questo notebook gestisce la pipeline completa per addestrare il modello di estrazione entità (NER / Token Classification) per **kochel**:
1. Setup dell'ambiente ed estrazione del dataset sintetico
2. Tokenizzazione e allineamento dei tag BIO con `all-MiniLM-L6-v2`
3. Fine-tuning su GPU con Hugging Face `Trainer`
4. Valutazione sul Test set non visto
5. Esportazione in grafo ONNX e quantizzazione dinamica INT8
6. Creazione dell'archivio compatto pronto per l'inferenza locale su CPU

## 📦 1. Setup Dipendenze ed Estrazione Dataset

In [ ]:
import os
from pathlib import Path

# 1. Installa tutte le librerie necessarie (inclusi onnxscript per il nuovo PyTorch e scikit-learn)
!pip install -q transformers datasets accelerate onnx onnxruntime onnxscript scikit-learn

# 2. Prepara la cartella data/
os.makedirs("data", exist_ok=True)

# 3. Estrai l'archivio se presente, altrimenti sposta eventuali jsonl caricati
if os.path.exists("kochel_data.zip"):
    !unzip -o kochel_data.zip -d data/
    print("kochel_data.zip estratto in data/")
elif os.path.exists("train.jsonl"):
    !mv *.jsonl *.json data/ 2>/dev/null || true
    print("File jsonl/json spostati in data/")

# 4. Verifica dei file pronti
print("\nVerifica file in data/:")
for f in ["train.jsonl", "val.jsonl", "test.jsonl", "labels.json"]:
    p = Path(f"data/{f}")
    if p.exists():
        print(f"  [OK] {f} ({p.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"  [ERRORE] Mancante: {f}")

## 📊 2. Tokenizer, Dataset & Preprocessing

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

# 1. Carica vocabolario delle label BIO
with open("data/labels.json") as f:
    label2id = json.load(f)
id2label = {int(v): k for k, v in label2id.items()}

# 2. Inizializza Tokenizer MiniLM
MODEL_CHECKPOINT = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# 3. Carica i dati JSONL
dataset = load_dataset(
    "json",
    data_files={
        "train": "data/train.jsonl",
        "validation": "data/val.jsonl",
        "test": "data/test.jsonl"
    }
)

def prepare_features(batch):
    input_ids_batch, labels_batch, attention_mask_batch = [], [], []
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    for tokens, tags in zip(batch["tokens"], batch["ner_tags"]):
        token_ids = tokenizer.convert_tokens_to_ids(tokens)[:126]
        tags = tags[:126]

        input_ids = [cls_id] + token_ids + [sep_id]
        ner_labels = [-100] + tags + [-100]
        
        input_ids_batch.append(input_ids)
        labels_batch.append(ner_labels)
        attention_mask_batch.append([1] * len(input_ids))

    return {
        "input_ids": input_ids_batch,
        "labels": labels_batch,
        "attention_mask": attention_mask_batch
    }

print("Elaborazione e tokenizzazione dataset...")
tokenized_datasets = dataset.map(
    prepare_features,
    batched=True,
    remove_columns=["text", "tokens", "ner_tags"]
)

# 4. Funzione di calcolo metriche (Accuracy, F1 ponderato, Precision, Recall)
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    flat_preds = []
    flat_labels = []

    for pred, lab in zip(predictions, labels):
        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                flat_preds.append(p_val)
                flat_labels.append(l_val)

    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_labels, flat_preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(flat_labels, flat_preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

print("Preprocessing completato.")

## 🧠 3. Addestramento e Valutazione Modello

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from pathlib import Path

# 1. Inizializzazione Modello per Token Classification
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Configurazione parametri di Training su GPU (Colab T4)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none"
)

# 3. Inizializzazione Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

print("Avvio addestramento su GPU...")
trainer.train()

# 4. Valutazione sul Test Set non visto
print("\n--- Risultati Test Set ---")
test_results = trainer.evaluate(tokenized_datasets["test"])
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# 5. Salvataggio del checkpoint addestrato e del tokenizer
EXPORT_DIR = Path("kochel_artifacts")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)
print(f"Modello e tokenizer salvati con successo in {EXPORT_DIR}/")

## ⚡ 4. Esportazione ONNX e Quantizzazione Dinamica INT8

In [ ]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification
from onnxruntime.quantization import quantize_dynamic, QuantType

EXPORT_DIR = Path("kochel_artifacts")

print("Caricamento modello su CPU per export ONNX...")
tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR)
model = AutoModelForTokenClassification.from_pretrained(EXPORT_DIR)
model.eval().to("cpu")

onnx_fp32 = EXPORT_DIR / "model.onnx"
onnx_int8 = EXPORT_DIR / "kochel_minilm_quantized.onnx"

# Input dummy per tracciare il grafo computazionale
dummy_ids = torch.tensor([[tokenizer.cls_token_id, 2000, 2001, tokenizer.sep_token_id]], dtype=torch.long)
dummy_mask = torch.tensor([[1, 1, 1, 1]], dtype=torch.long)

print("Esportazione grafo ONNX...")
torch.onnx.export(
    model,
    (dummy_ids, dummy_mask),
    str(onnx_fp32),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size", 1: "sequence_length"}
    },
    opset_version=14,
    dynamo=False
)

print("Quantizzazione dinamica in INT8...")
quantize_dynamic(
    model_input=str(onnx_fp32),
    model_output=str(onnx_int8),
    weight_type=QuantType.QInt8
)

# Rimuovi il modello FP32 non quantizzato per risparmiare spazio
if onnx_fp32.exists():
    onnx_fp32.unlink()

print(f"Modello INT8 generato con successo: {onnx_int8} ({onnx_int8.stat().st_size / (1024*1024):.1f} MB)")

## 📦 5. Impacchettamento Archivio e Download

In [ ]:
import shutil

# 1. Assicura che labels.json sia presente tra gli artefatti
shutil.copy("data/labels.json", "kochel_artifacts/labels.json")

# 2. Crea archivio flat (-j) con modello ONNX, vocabolario e tokenizer
!zip -j -o kochel_model_ready.zip \
    kochel_artifacts/kochel_minilm_quantized.onnx \
    kochel_artifacts/labels.json \
    kochel_artifacts/tokenizer.json \
    kochel_artifacts/tokenizer_config.json \
    kochel_artifacts/special_tokens_map.json \
    kochel_artifacts/vocab.txt

print("\n--- Contenuto archivio kochel_model_ready.zip ---")
!unzip -l kochel_model_ready.zip

# 3. Avvia download (su Google Colab) o messaggio di conferma
try:
    from google.colab import files
    print("\nAvvio download diretto...")
    files.download("kochel_model_ready.zip")
except Exception as e:
    print("\nPuoi scaricare 'kochel_model_ready.zip' dalla barra laterale sinistra dei file di Colab.")